In [0]:
from pyspark.sql.functions import (
    col, when, year, month, dayofmonth, hour, dayofweek, date_format,
    unix_timestamp, md5, concat_ws, row_number, lit
)
from pyspark.sql.window import Window

# Read from bronze table
df_bronze = spark.table("workspace.taxi_schema.taxi_bronze")

# Data Quality Filters - Remove bad rows
df_filtered = df_bronze.filter(
    # Critical fields must not be null
    col("pickup_datetime").isNotNull() &
    col("dropoff_datetime").isNotNull() &
    col("PULocationID").isNotNull() &
    col("DOLocationID").isNotNull() &
    col("trip_distance").isNotNull() &
    col("total_amount").isNotNull() &
    
    # Dropoff must be after pickup
    (col("dropoff_datetime") > col("pickup_datetime")) &
    
    # Amounts should be non-negative (allow zero for promotions/adjustments)
    (col("fare_amount") >= 0) &
    (col("total_amount") >= 0) &
    (col("trip_distance") >= 0) &
    
    # Reasonable trip duration (5 seconds to 24 hours)
    ((unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) >= 5) &
    ((unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) <= 86400) &
    
    # Valid passenger count (0 to 9, allowing 0 for special cases)
    (col("passenger_count").isNull() | ((col("passenger_count") >= 0) & (col("passenger_count") <= 9))) &
    
    # Valid payment type (1-5 are typical)
    (col("payment_type").isNull() | ((col("payment_type") >= 1) & (col("payment_type") <= 5)))
)

# Add derived columns
df_enriched = df_filtered.withColumn(
    # Trip duration in minutes
    "trip_duration_minutes",
    (unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60
).withColumn(
    # Date dimensions
    "pickup_year", year(col("pickup_datetime"))
).withColumn(
    "pickup_month", month(col("pickup_datetime"))
).withColumn(
    "pickup_day", dayofmonth(col("pickup_datetime"))
).withColumn(
    "pickup_hour", hour(col("pickup_datetime"))
).withColumn(
    "pickup_day_of_week", dayofweek(col("pickup_datetime"))
).withColumn(
    "pickup_day_name", date_format(col("pickup_datetime"), "EEEE")
).withColumn(
    # Business flags
    "is_airport_trip", when(col("Airport_fee").isNotNull() & (col("Airport_fee") > 0), True).otherwise(False)
).withColumn(
    # Cast types where needed (ensure integer types for IDs)
    "VendorID", col("VendorID").cast("int")
).withColumn(
    "passenger_count", col("passenger_count").cast("int")
).withColumn(
    "payment_type", col("payment_type").cast("int")
).withColumn(
    "RatecodeID", col("RatecodeID").cast("int")
)

# Create business key for deduplication
df_with_key = df_enriched.withColumn(
    "_business_key",
    md5(concat_ws("|",
        col("taxi_type"),
        col("pickup_datetime"),
        col("dropoff_datetime"),
        col("PULocationID"),
        col("DOLocationID"),
        col("VendorID"),
        col("trip_distance")
    ))
)

# Deduplicate - keep most recently ingested record per business key
window_spec = Window.partitionBy("_business_key").orderBy(col("_ingested_at").desc())
df_deduplicated = (
    df_with_key
    .withColumn("_row_num", row_number().over(window_spec))
    .filter(col("_row_num") == 1)
    .drop("_row_num", "_business_key")
)

# Write to silver Delta table with partitioning
(
    df_deduplicated
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("pickup_year", "pickup_month")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.taxi_schema.taxi_silver")
)

print(f"Silver table created with {df_deduplicated.count()} records")